# 04 — Predykcja: GP Meksyku 2026

**Cel notebooka:** zamienić przewidziane pozycje z `03_model` na gotową prognozę (zwycięzca, podium, top 10, najszybsze okrążenie) i **zamrozić ją** w pliku, zanim wyścig się zacznie.

Wczytujemy wyłącznie `data/model_prediction_mexico.csv` z `03_model` — modelu tu nie liczymy od nowa.

> ⚠️ Prognozę zamrażamy **przed startem** (20:00 UTC) i od razu commitujemy. Po wyścigu niczego tu nie zmieniamy.

## 4.1 Zwycięzca, podium, top 10

**Co robimy:** wczytujemy przewidziane pozycje, sortujemy kierowców od najniższej `pred_pos` i wypisujemy trzy listy.

**Dlaczego:** to nasze trzy pierwsze cele. Zwycięzca to kierowca z najniższą przewidzianą pozycją, podium to trzech pierwszych, a top 10 to dziesięciu pierwszych. Nie ma tu losowania ani dodatkowych reguł, to po prostu kolejność z modelu.

In [3]:
import pandas as pd

pred = pd.read_csv("data/model_prediction_mexico.csv").sort_values("pred_pos").reset_index(drop=True)
pred.index += 1

print("Zwycięzca:", pred.loc[1, "name_acronym"])
print("Podium:   ", ", ".join(pred.loc[1:3, "name_acronym"]))
print("Top 10:   ", ", ".join(pred.loc[1:10, "name_acronym"]))
pred.loc[1:10, ["name_acronym", "team_name", "start_pos", "pred_pos"]].round(2)

Zwycięzca: HAM
Podium:    HAM, RUS, LEC
Top 10:    HAM, RUS, LEC, ANT, VER, NOR, HAD, LIN, GAS, PIA


,name_acronym,team_name,start_pos,pred_pos
1,HAM,Ferrari,1.0,2.30
2,RUS,Mercedes,2.0,3.00
3,LEC,Ferrari,3.0,3.17
4,ANT,Mercedes,5.0,3.95
5,VER,Red Bull Racing,4.0,4.03
6,NOR,McLaren,6.0,5.27
7,HAD,Red Bull Racing,7.0,6.36
8,LIN,Racing Bulls,8.0,8.21
9,GAS,Alpine,9.0,8.28
10,PIA,McLaren,12.0,8.75


## 4.2 Najszybsze okrążenie

**Co robimy:** wybieramy kierowcę z najlepszym dotychczasowym tempem wyścigowym (`driver_pace` z `03_model`) i pokazujemy kilku najbliższych kandydatów.

**Dlaczego:** w 3.12 ta reguła trafiała najszybsze okrążenie w 5 z 15 wyścigów (33%), czyli znacznie lepiej niż przypadek (około 5%), tak samo jak „najczęstszy dotychczas”, i lepiej niż pole position (4 z 15). To mało pewna prognoza, ale najlepsza, jaką mamy. Najszybsze okrążenie jedzie często ktoś, kto zjechał po świeże opony na końcu wyścigu, więc zawsze pozostaje loterią.

In [4]:
fastest = pred.sort_values("driver_pace").reset_index(drop=True)
fastest.index += 1

print("Najszybsze okrążenie:", fastest.loc[1, "name_acronym"])
fastest.loc[1:5, ["name_acronym", "team_name", "driver_pace"]].round(2)

Najszybsze okrążenie: ANT


,name_acronym,team_name,driver_pace
1,ANT,Mercedes,-1.76
2,LEC,Ferrari,-1.55
3,HAM,Ferrari,-1.42
4,VER,Red Bull Racing,-1.33
5,RUS,Mercedes,-1.30


## 4.3 Czas najszybszego okrążenia

**Co robimy:** szacujemy czas najszybszego okrążenia wyścigu jako **czas pole position z dzisiejszych kwalifikacji × typowy stosunek** (najszybsze okrążenie wyścigu / czas pole) z 18 wyścigów 2026.

**Dlaczego tak:** historii toru (2023–2025) nie da się użyć, bo samochody z 2026 są inne. Czas pole z dzisiaj mamy, z tymi samymi samochodami, a stosunek do najszybszego okrążenia wyścigu jest w sezonie dość stabilny (mediana 1,042, odchylenie 0,015). Przedział błędu: typowo około ±1,2 s, zakres historyczny 1,020–1,073.

In [5]:
quali = pd.read_csv("data/quali_2026.csv")
fl = pd.read_csv("data/fastest_laps_2026.csv").set_index("round")["fastest_lap_time"]

pole_time = quali[quali["position"] == 1].groupby("round")["best_time"].min()
ratio = (fl / pole_time.reindex(fl.index)).dropna()

pole_mexico = pole_time.loc[19]
fl_estimate = pole_mexico * ratio.median()
fl_low, fl_high = pole_mexico * ratio.min(), pole_mexico * ratio.max()

print(f"Pole w Meksyku: {pole_mexico:.3f} s")
print(f"Stosunek najszybsze okrążenie / pole: mediana {ratio.median():.3f}, zakres {ratio.min():.3f}–{ratio.max():.3f} (n = {len(ratio)})")
print(f"Szacunek najszybszego okrążenia: {fl_estimate:.2f} s (zakres historyczny {fl_low:.2f}–{fl_high:.2f} s)")

Pole w Meksyku: 76.514 s
Stosunek najszybsze okrążenie / pole: mediana 1.042, zakres 1.020–1.073 (n = 18)
Szacunek najszybszego okrążenia: 79.71 s (zakres historyczny 78.03–82.09 s)


## 4.4 Zamrożenie prognozy

**Co robimy:** zapisujemy całą prognozę do dwóch plików CSV razem ze znacznikiem czasu (UTC) i od razu je commitujemy — **przed startem wyścigu**.

- `data/prediction_mexico_2026.csv` — kolejność wszystkich 22 kierowców z flagami zwycięzcy, podium i top 10,
- `data/prediction_mexico_2026_fastest_lap.csv` — przewidywany kierowca i czas najszybszego okrążenia.

**Dlaczego:** żeby po wyścigu można było uczciwie sprawdzić, co przewidzieliśmy, bez możliwości poprawiania.

In [6]:
frozen_at = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S UTC")

final = pred[["name_acronym", "team_name", "start_pos", "pred_pos"]].copy()
final.insert(0, "predicted_position", final.index)
final["winner"] = final["predicted_position"] == 1
final["podium"] = final["predicted_position"] <= 3
final["top10"] = final["predicted_position"] <= 10
final["frozen_at"] = frozen_at
final.round(2).to_csv("data/prediction_mexico_2026.csv", index=False)

pd.DataFrame([{
    "fastest_lap_driver": fastest.loc[1, "name_acronym"],
    "fastest_lap_time_estimate_s": round(fl_estimate, 2),
    "range_low_s": round(fl_low, 2),
    "range_high_s": round(fl_high, 2),
    "frozen_at": frozen_at,
}]).to_csv("data/prediction_mexico_2026_fastest_lap.csv", index=False)

print("Zamrożono:", frozen_at)
final.loc[1:10, ["predicted_position", "name_acronym", "team_name", "start_pos", "winner", "podium", "top10"]]

Zamrożono: 2026-11-01 16:36:40 UTC


,predicted_position,name_acronym,team_name,start_pos,winner,podium,top10
1,1,HAM,Ferrari,1.0,True,True,True
2,2,RUS,Mercedes,2.0,False,True,True
3,3,LEC,Ferrari,3.0,False,True,True
4,4,ANT,Mercedes,5.0,False,False,True
5,5,VER,Red Bull Racing,4.0,False,False,True
6,6,NOR,McLaren,6.0,False,False,True
7,7,HAD,Red Bull Racing,7.0,False,False,True
8,8,LIN,Racing Bulls,8.0,False,False,True
9,9,GAS,Alpine,9.0,False,False,True
10,10,PIA,McLaren,12.0,False,False,True


## 4.5 Opis modelu, zmiennych i ograniczeń

*(Dopisane po zamrożeniu prognozy. Opis nie zmienia żadnych liczb ani plików z prognozą.)*

### Model
**Ridge** (regresja liniowa z karą L2, `alpha = 1.0`, scikit-learn). Przewiduje **pozycję na mecie** kierowcy. Kierowców układamy po przewidzianej pozycji: najniższa = zwycięzca, trzech pierwszych = podium, dziesięciu pierwszych = top 10.
- **Dane treningowe:** 18 zakończonych wyścigów 2026, tylko kierowcy, którzy **ukończyli** wyścig (awaria nie mówi nic o tempie).
- **Wyuczone współczynniki:** pozycja na mecie ≈ 3,8 + 0,53 · `start_pos` + 1,43 · `driver_pace`.
- **Walidacja:** walk-forward, rundy 4–18. Dla każdej rundy model uczony tylko na wcześniejszych, oceniany na tej jednej.

### Zmienne użyte w modelu
1. **`start_pos` — pozycja startowa.** Pozycja w kwalifikacjach (zastępuje pozycję startową, bo endpoint `starting_grid` jest w 2026 pusty). Brak czasu w kwalifikacjach = 22 (koniec stawki; w Meksyku dotyczy LAW, który ma karę).
2. **`driver_pace` — tempo wyścigowe kierowcy z poprzednich wyścigów (w %, ujemne = szybciej niż środek stawki).** Powstaje w dwóch krokach:
   - *tempo w jednym wyścigu (`pace_pct`):* z okrążeń wyścigu odrzucamy 1. okrążenie, okrążenia wyjazdowe i zjazdowe z alei, okrążenia za samochodem bezpieczeństwa (> 107% mediany wyścigu) i braki; kierowcy z mniej niż 10 czystymi okrążeniami odpadają. Tempo = mediana czasów czystych okrążeń kierowcy, podzielona przez medianę tych median w wyścigu, minus 1, razy 100.
   - *wartość dla przewidywanej rundy:* suma `pace_pct` kierowcy z **wcześniejszych** rund podzielona przez (liczba wyścigów + 2). „+ 2” to skurczenie do zera przy małej próbie. Brak historii (np. HIR) = 0.

### Co sprawdziliśmy i odrzuciliśmy (nie poprawiało walidacji)
strata do pole w % (`gap`), umiejętność kierowcy (zysk miejsc ponad typowy dla pozycji startowej, `skill`), awaryjność kierowcy i zespołu, tempo zespołu (`team_pace`). Dodatkowo las losowy i gradient boosting wypadły gorzej niż Ridge (zwycięzca 47% i 53% wobec 73%).

### Najszybsze okrążenie
- **Kierowca:** ten z najlepszym `driver_pace` (ANT).
- **Czas:** czas pole z dzisiejszych kwalifikacji × mediana stosunku „najszybsze okrążenie wyścigu / czas pole” z 18 wyścigów (1,042).

### Wyniki walidacji (rundy 4–18, 15 wyścigów)
| | Zwycięzca | Top 3 | Top 10 | Błąd pozycji |
|---|---|---|---|---|
| Reguła „jak na starcie” | 67% | 58% | 78% | 2,20 |
| Ridge: `start_pos` + `driver_pace` | 73% | 62% | 78% | 2,06 |

### Ograniczenia
- Tylko 15 wyścigów do oceny: różnica jednego trafienia zwycięzcy to szum, nie dowód przewagi.
- Podium ma tę samą trójkę co reguła „jak na starcie”; zysk widać głównie w środku stawki.
- Top 3 i top 10 oceniane jako **wystąpienie** na liście, a nie kolejność wewnątrz listy.
- Najszybsze okrążenie to najmniej pewny cel (około 33% trafień w walidacji); szacunek czasu ma typowy błąd ok. ±1,2 s.
- Model nie zna pogody, samochodu bezpieczeństwa, kar poza pozycją startową ani strategii.